# Anti-bandwidth Problem SAT Encoder

Notebook self-contained for Kaggle. It includes the graph parser, SAT core, SCE exactly-one encoding, ladder AMK encoding, ABP model, binary search, validation, and benchmark runners.

Input format: the first numeric line has `n unused m`; `n` is vertex count and `m` is edge count. Remaining lines are undirected edges.

In [ ]:
# Kaggle usually allows internet in notebooks when enabled.
# Run this once if python-sat is not already installed.
try:
    from pysat.solvers import Glucose4
    print('python-sat is available')
except ImportError:
    %pip -q install python-sat
    from pysat.solvers import Glucose4
    print('python-sat installed')

In [ ]:
from pathlib import Path
from time import perf_counter
import os
import signal

from pysat.solvers import Glucose4

In [ ]:
class VarPool:
    def __init__(self):
        self.next_id = 1
        self.key_to_id = {}
        self.id_to_key = {}

    def var(self, *key):
        if key not in self.key_to_id:
            vid = self.next_id
            self.next_id += 1
            self.key_to_id[key] = vid
            self.id_to_key[vid] = key
        return self.key_to_id[key]

    def explain(self, lit):
        sign = '' if lit > 0 else 'not '
        return f'{sign}{self.id_to_key[abs(lit)]}'

    @property
    def num_vars(self):
        return self.next_id - 1


class CNF:
    def __init__(self):
        self.clauses = []

    def add_clause(self, *lits):
        self.clauses.append(list(lits))

    def __len__(self):
        return len(self.clauses)


class EncodingContext:
    def __init__(self):
        self.pool = VarPool()
        self.cnf = CNF()

    def var(self, *key):
        return self.pool.var(*key)

    def add_clause(self, *lits):
        self.cnf.add_clause(*lits)

    def explain(self, lit):
        return self.pool.explain(lit)

    @property
    def clauses(self):
        return self.cnf.clauses

    @property
    def num_vars(self):
        return self.pool.num_vars

    @property
    def num_clauses(self):
        return len(self.cnf)

In [ ]:
class SCE:
    def __init__(self, ctx, name='sce'):
        self.ctx = ctx
        self.name = name
        self.encoding_id = 0

    def add_clause(self, clause):
        self.ctx.add_clause(*clause)

    def next_encoding_id(self):
        self.encoding_id += 1
        return self.encoding_id

    def new_counter_states(self, variables, K):
        registers = {}
        encoding_id = self.next_encoding_id()
        for i in range(len(variables) - 1):
            for j in range(min(i + 1, K)):
                registers[(i, j)] = self.ctx.var('sce_r', self.name, encoding_id, i, j + 1)
        return registers

    def amk(self, variables, K):
        variables = list(variables)
        n = len(variables)
        if K < 0:
            self.add_clause([])
            return {}
        if K >= n:
            return {}
        if K == 0:
            for x in variables:
                self.add_clause([-x])
            return {}

        R = self.new_counter_states(variables, K)
        for i in range(n - 1):
            self.add_clause([-variables[i], R[(i, 0)]])
        for i in range(1, n - 1):
            for j in range(min(i, K)):
                self.add_clause([-R[(i - 1, j)], R[(i, j)]])
        for i in range(1, n - 1):
            for j in range(1, min(i + 1, K)):
                self.add_clause([-variables[i], -R[(i - 1, j - 1)], R[(i, j)]])
        for i in range(K, n):
            self.add_clause([-variables[i], -R[(i - 1, K - 1)]])
        return R

    def alk(self, variables, K):
        variables = list(variables)
        n = len(variables)
        if K <= 0:
            return {}
        if K > n:
            self.add_clause([])
            return {}
        if K == n:
            for x in variables:
                self.add_clause([x])
            return {}

        R = self.new_counter_states(variables, K)
        for i in range(n - 1):
            self.add_clause([-variables[i], R[(i, 0)]])
        for i in range(1, n - 1):
            for j in range(min(i, K)):
                self.add_clause([-R[(i - 1, j)], R[(i, j)]])
        for i in range(1, n - 1):
            for j in range(1, min(i + 1, K)):
                self.add_clause([-variables[i], -R[(i - 1, j - 1)], R[(i, j)]])
        for i in range(1, n - 1):
            for j in range(min(i, K)):
                self.add_clause([variables[i], R[(i - 1, j)], -R[(i, j)]])
        for i in range(min(K, n - 1)):
            self.add_clause([variables[i], -R[(i, i)]])
        for i in range(1, n - 1):
            for j in range(1, min(i + 1, K)):
                self.add_clause([R[(i - 1, j - 1)], -R[(i, j)]])

        last_x = variables[-1]
        last_prefix = n - 2
        if K == 1:
            self.add_clause([R[(last_prefix, 0)], last_x])
        else:
            self.add_clause([R[(last_prefix, K - 1)], last_x])
            self.add_clause([R[(last_prefix, K - 1)], R[(last_prefix, K - 2)]])
        return R

    def exk(self, variables, K):
        self.alk(variables, K)
        self.amk(variables, K)

In [ ]:
class LadderAMK:
    def __init__(self, variables, w, K, name='ladder_amk'):
        self.variables = list(variables)
        self.w = w
        self.K = K
        self.name = name
        self.ctx = None
        self.subsets = None
        self._R = None

    def encode(self, ctx):
        if not 1 < self.w <= len(self.variables):
            raise ValueError('w must be in [2, len(variables)]')
        if not 1 <= self.K < self.w:
            raise ValueError('K must be in [1, w - 1]')
        self.ctx = ctx
        self.subsets = self.create_subsets()
        self._R = self.create_blocks()
        self.connect_blocks()
        return self

    def add_clause(self, clause):
        self.ctx.add_clause(*clause)

    def new_register(self, block_id, row, col):
        return self.ctx.var('ladder', self.name, block_id, row, col)

    def R(self, block_id, row, col):
        if self._R is None:
            raise RuntimeError('encode(ctx) must be called before R')
        block = self._R[block_id]
        return block[row][col]

    def create_subsets(self):
        subsets = [[0]]
        variables = [0] + self.variables
        n = len(variables) - 1
        for i in range(1, n + 1, self.w):
            subsets.append(variables[i:min(i + self.w, n + 1)])
        return subsets

    def create_block(self, block_id, subset, is_AMK=True):
        R = [[0 for _ in range(self.K + 1)]]
        x = [0] + subset
        w_i = len(x) - 1
        limit = w_i + 1 if w_i < self.w else self.w

        for row in range(1, limit):
            register = [0]
            for col in range(1, min(row, self.K) + 1):
                register.append(self.new_register(block_id, row, col))
            R.append(register)

        for row in range(1, limit):
            self.add_clause([-x[row], R[row][1]])
        for row in range(2, limit):
            for col in range(1, min(row - 1, self.K) + 1):
                self.add_clause([-R[row - 1][col], R[row][col]])
        for row in range(2, limit):
            for col in range(2, min(row, self.K) + 1):
                self.add_clause([-x[row], -R[row - 1][col - 1], R[row][col]])
        for row in range(1, min(self.K, w_i) + 1):
            self.add_clause([x[row], -R[row][row]])
        for row in range(2, limit):
            for col in range(2, min(row, self.K) + 1):
                self.add_clause([R[row - 1][col - 1], -R[row][col]])
        for row in range(2, limit):
            for col in range(1, min(row - 1, self.K) + 1):
                self.add_clause([x[row], R[row - 1][col], -R[row][col]])
        if is_AMK:
            for row in range(self.K + 1, w_i + 1):
                self.add_clause([-x[row], -R[row - 1][self.K]])
        return R

    def create_blocks(self):
        blocks = [None]
        if len(self.subsets) == 2:
            blocks.append(self.create_block(1, list(reversed(self.subsets[1])), is_AMK=True))
            return blocks
        blocks.append(self.create_block(1, list(reversed(self.subsets[1])), is_AMK=True))
        for i in range(2, len(self.subsets) - 1):
            lr_subset = self.subsets[i].copy()
            blocks.append(self.create_block(len(blocks), lr_subset, is_AMK=True))
            blocks.append(self.create_block(len(blocks), list(reversed(lr_subset)), is_AMK=False))
        blocks.append(self.create_block(len(blocks), self.subsets[-1], is_AMK=True))
        return blocks

    def block_to_subset_index(self, block_index):
        return (block_index + 2) // 2

    def connect_blocks(self):
        m = len(self.subsets) - 1
        for block in range(1, 2 * (m - 1) + 1, 2):
            rl_subset_index = self.block_to_subset_index(block)
            lr_subset_index = self.block_to_subset_index(block + 1)
            w_i = min(len(self.subsets[lr_subset_index]) + 1, len(self.subsets[rl_subset_index]))
            for row in range(2, w_i + 1):
                for p in range(1, self.K + 1):
                    left_len = self.w - row + 1
                    right_len = row - 1
                    left_threshold = self.K - p + 1
                    right_threshold = p
                    if left_threshold <= left_len and right_threshold <= right_len:
                        self.add_clause([-self.R(block, left_len, left_threshold), -self.R(block + 1, right_len, right_threshold)])


def ladder_at_most_k(ctx, variables, w, K, name='ladder_amk'):
    return LadderAMK(variables, w, K, name=name).encode(ctx)

In [ ]:
class Graph:
    def __init__(self, n, edges):
        if n <= 0:
            raise ValueError('n must be positive')
        self.n = n
        self.V = list(range(1, n + 1))
        self.E = []
        seen = set()
        for u, v in edges:
            if u == v:
                raise ValueError('self-loops are not supported')
            if not 1 <= u <= n or not 1 <= v <= n:
                raise ValueError('edge endpoint is outside vertex range')
            edge = (u, v) if u < v else (v, u)
            if edge not in seen:
                seen.add(edge)
                self.E.append(edge)

    @classmethod
    def from_rnd(cls, path):
        path = Path(path)
        lines = []
        for line in path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if line and not line.startswith('Nombre del problema'):
                lines.append(line)
        n, _, edge_count = map(int, lines[0].split()[:3])
        edges = [tuple(map(int, line.split()[:2])) for line in lines[1:]]
        if len(edges) != edge_count:
            raise ValueError(f'expected {edge_count} edges, got {len(edges)}')
        return cls(n, edges)


class AntiBandwidthProblem:
    def __init__(self, graph, bandwidth, ctx=None, symmetry_break=True, anchor=None):
        if bandwidth <= 0:
            raise ValueError('bandwidth must be positive')
        self.graph = graph
        self.b = bandwidth
        self.ctx = ctx or EncodingContext()
        self.sce = SCE(self.ctx, name='abp_sce')
        self.symmetry_break = symmetry_break
        self.anchor = anchor if anchor is not None else graph.V[0]
        if self.anchor not in graph.V:
            raise ValueError('anchor must be a graph vertex')
        self.x = {(i, l): self.ctx.var('abp_x', i, l) for i in graph.V for l in range(1, graph.n + 1)}
        self.ladders = {}

    def add_clause(self, *clause):
        self.ctx.add_clause(*clause)

    def X(self, i, l):
        return self.x[(i, l)]

    def labels_of(self, i):
        return [self.X(i, l) for l in range(1, self.graph.n + 1)]

    def vertices_at(self, l):
        return [self.X(i, l) for i in self.graph.V]

    def add_permutation(self):
        for i in self.graph.V:
            self.sce.exk(self.labels_of(i), 1)
        for l in range(1, self.graph.n + 1):
            self.sce.exk(self.vertices_at(l), 1)

    def add_symmetry_breaking(self):
        # Label reversal is symmetric: l -> n + 1 - l.
        # Keep only solutions where the anchor vertex is in the left half.
        max_anchor_label = (self.graph.n + 1) // 2
        for label in range(max_anchor_label + 1, self.graph.n + 1):
            self.add_clause(-self.X(self.anchor, label))

    def add_ladders(self):
        if self.b > self.graph.n:
            self.add_clause()
            return
        for i in self.graph.V:
            self.ladders[i] = ladder_at_most_k(self.ctx, self.labels_of(i), self.b, 1, name=f'abp_{i}')

    def subset_count(self):
        return (self.graph.n + self.b - 1) // self.b

    def subset_bounds(self, subset):
        start = (subset - 1) * self.b + 1
        return start, min(start + self.b - 1, self.graph.n)

    def prefix_block(self, subset):
        return None if subset == 1 else 2 * subset - 2

    def suffix_block(self, subset):
        return None if subset == self.subset_count() else 2 * subset - 1

    def zero_atom(self, i, block, length):
        if block is None:
            return None
        try:
            return -self.ladders[i].R(block, length, 1)
        except (IndexError, ValueError):
            return None

    def zero_atoms(self, i, start, length):
        end = start + length - 1
        if not 1 <= start <= end <= self.graph.n:
            raise ValueError('interval is outside label range')
        atoms = []
        label = start
        while label <= end:
            subset = (label - 1) // self.b + 1
            subset_start, subset_end = self.subset_bounds(subset)
            seg_end = min(end, subset_end)
            seg_len = seg_end - label + 1
            atom = None
            if label == subset_start:
                atom = self.zero_atom(i, self.prefix_block(subset), seg_len)
            if atom is None and seg_end == subset_end:
                atom = self.zero_atom(i, self.suffix_block(subset), seg_len)
            if atom is None:
                atoms.extend(-self.X(i, l) for l in range(label, seg_end + 1))
            else:
                atoms.append(atom)
            label = seg_end + 1
        return atoms

    def add_edges(self):
        if self.b <= 1:
            return
        if self.b > self.graph.n:
            if self.graph.E:
                self.add_clause()
            return
        for u, v in self.graph.E:
            for start in range(1, self.graph.n - self.b + 2):
                u_zero = self.zero_atoms(u, start, self.b)
                v_zero = self.zero_atoms(v, start, self.b)
                for a in u_zero:
                    for c in v_zero:
                        self.add_clause(a, c)

    def encode(self):
        self.add_permutation()
        if self.symmetry_break:
            self.add_symmetry_breaking()
        if self.b > 1:
            self.add_ladders()
            self.add_edges()
        return self

    def solve(self):
        with Glucose4(bootstrap_with=self.ctx.clauses) as solver:
            if not solver.solve():
                return None
            model = {lit for lit in solver.get_model() if lit > 0}
        return {i: next(l for l in range(1, self.graph.n + 1) if self.X(i, l) in model) for i in self.graph.V}


def validate_labeling(graph, bandwidth, labeling):
    if labeling is None:
        return False
    labels = [labeling.get(i) for i in graph.V]
    if sorted(labels) != list(range(1, graph.n + 1)):
        return False
    return all(abs(labeling[u] - labeling[v]) >= bandwidth for u, v in graph.E)


def solve_decision(graph, bandwidth, symmetry_break=True, anchor=None):
    abp = AntiBandwidthProblem(graph, bandwidth, symmetry_break=symmetry_break, anchor=anchor).encode()
    labeling = abp.solve()
    return labeling, abp.ctx.num_vars, abp.ctx.num_clauses


def find_max_bandwidth(graph, lower_bound=1, upper_bound=None, symmetry_break=True, anchor=None):
    if upper_bound is None:
        upper_bound = graph.n
    if not 1 <= lower_bound <= upper_bound <= graph.n:
        raise ValueError('bounds must satisfy 1 <= lower_bound <= upper_bound <= n')
    best_b = 0
    best_labeling = None
    low, high = lower_bound, upper_bound
    while low <= high:
        mid = (low + high) // 2
        labeling, _, _ = solve_decision(graph, mid, symmetry_break=symmetry_break, anchor=anchor)
        if labeling is None:
            high = mid - 1
        else:
            best_b = mid
            best_labeling = labeling
            low = mid + 1
    return best_b, best_labeling


def solve_decision_direct(graph, bandwidth):
    """Reference encoding: forbid every edge-label pair with distance < bandwidth."""
    ctx = EncodingContext()
    sce = SCE(ctx, name='direct_sce')
    x = {(i, l): ctx.var('direct_x', i, l) for i in graph.V for l in range(1, graph.n + 1)}

    for i in graph.V:
        sce.exk([x[(i, l)] for l in range(1, graph.n + 1)], 1)
    for l in range(1, graph.n + 1):
        sce.exk([x[(i, l)] for i in graph.V], 1)

    for u, v in graph.E:
        for a in range(1, graph.n + 1):
            lo = max(1, a - bandwidth + 1)
            hi = min(graph.n, a + bandwidth - 1)
            for c in range(lo, hi + 1):
                ctx.add_clause(-x[(u, a)], -x[(v, c)])

    with Glucose4(bootstrap_with=ctx.clauses) as solver:
        if not solver.solve():
            return None, ctx.num_vars, ctx.num_clauses
        model = {lit for lit in solver.get_model() if lit > 0}

    labeling = {i: next(l for l in range(1, graph.n + 1) if x[(i, l)] in model) for i in graph.V}
    return labeling, ctx.num_vars, ctx.num_clauses


def find_max_bandwidth_direct(graph, lower_bound=1, upper_bound=None):
    if upper_bound is None:
        upper_bound = graph.n
    if not 1 <= lower_bound <= upper_bound <= graph.n:
        raise ValueError('bounds must satisfy 1 <= lower_bound <= upper_bound <= n')
    best_b = 0
    best_labeling = None
    low, high = lower_bound, upper_bound
    while low <= high:
        mid = (low + high) // 2
        labeling, _, _ = solve_decision_direct(graph, mid)
        if labeling is None:
            high = mid - 1
        else:
            best_b = mid
            best_labeling = labeling
            low = mid + 1
    return best_b, best_labeling


def compare_ladder_with_direct(graph, bandwidths=None):
    if bandwidths is None:
        bandwidths = range(1, graph.n + 1)
    rows = []
    for b in bandwidths:
        t0 = perf_counter()
        ladder_labeling, ladder_vars, ladder_clauses = solve_decision(graph, b)
        t1 = perf_counter()
        direct_labeling, direct_vars, direct_clauses = solve_decision_direct(graph, b)
        t2 = perf_counter()
        row = {
            'b': b,
            'ladder_sat': ladder_labeling is not None,
            'direct_sat': direct_labeling is not None,
            'match': (ladder_labeling is not None) == (direct_labeling is not None),
            'ladder_valid': validate_labeling(graph, b, ladder_labeling),
            'direct_valid': validate_labeling(graph, b, direct_labeling),
            'ladder_vars': ladder_vars,
            'direct_vars': direct_vars,
            'ladder_clauses': ladder_clauses,
            'direct_clauses': direct_clauses,
            'ladder_time': t1 - t0,
            'direct_time': t2 - t1,
        }
        rows.append(row)
        print(row, flush=True)
    return rows

In [ ]:
# Quick unit-style checks.
g = Graph(8, [(1, 2)])
abp = AntiBandwidthProblem(g, 4).encode()
print([abp.ctx.explain(atom) for atom in abp.zero_atoms(1, 2, 4)])
assert len(abp.zero_atoms(1, 2, 4)) == 2

g_small = Graph(4, [(1, 2)])
b_small, labeling_small = find_max_bandwidth(g_small)
print('small best:', b_small, labeling_small)
assert validate_labeling(g_small, b_small, labeling_small)

# Cross-check ladder encoding against direct pairwise encoding on a small graph.
rows = compare_ladder_with_direct(g_small)
assert all(row['match'] for row in rows)
b_direct, labeling_direct = find_max_bandwidth_direct(g_small)
assert b_small == b_direct
assert validate_labeling(g_small, b_direct, labeling_direct)

## Load benchmark files

On Kaggle, upload the `.rnd` files as a Dataset, then set `DATA_DIR` to the mounted folder, for example `/kaggle/input/abp-benchmarks`.

In [ ]:
DATA_DIR = Path('/kaggle/input/abp-benchmarks')

# Local fallback when running inside this repository.
if not DATA_DIR.exists() and Path('datas').exists():
    DATA_DIR = Path('datas')

files = sorted(DATA_DIR.rglob('*.rnd'))

# Fill these from the paper/benchmark table when available.
# Format: 'file_name.rnd': (lower_bound, upper_bound)
BENCHMARK_BOUNDS = {
    # 'A-pores_1.mtx.rnd': (1, 30),
}

def bounds_for(path, graph):
    return BENCHMARK_BOUNDS.get(path.name, (1, graph.n))

print('DATA_DIR =', DATA_DIR)
print('files =', len(files))
for path in files[:10]:
    print(path)

In [ ]:
# Run one benchmark.
path = files[0]
t0 = perf_counter()
graph = Graph.from_rnd(path)
lb, ub = bounds_for(path, graph)
best_b, labeling = find_max_bandwidth(graph, lb, ub)
print(path.name, 'n=', graph.n, 'm=', len(graph.E), 'bounds=', (lb, ub), 'best=', best_b, 'valid=', validate_labeling(graph, best_b, labeling), 'time=', round(perf_counter() - t0, 2))
print(labeling)

In [ ]:
# Run all files. Use this on Kaggle; large instances may take time.
results = []
for path in files:
    t0 = perf_counter()
    graph = Graph.from_rnd(path)
    lb, ub = bounds_for(path, graph)
    try:
        best_b, labeling = find_max_bandwidth(graph, lb, ub)
        valid = validate_labeling(graph, best_b, labeling)
        elapsed = perf_counter() - t0
        row = {'file': path.name, 'n': graph.n, 'm': len(graph.E), 'lb': lb, 'ub': ub, 'best': best_b, 'valid': valid, 'time': elapsed}
    except Exception as exc:
        elapsed = perf_counter() - t0
        row = {'file': path.name, 'n': graph.n, 'm': len(graph.E), 'lb': lb, 'ub': ub, 'best': None, 'valid': False, 'time': elapsed, 'error': repr(exc)}
    results.append(row)
    print(row, flush=True)

results